In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import pandas as pd
import sumy
import os

CNN/dailymail dataset

In [22]:
df = pd.read_csv("data/test.csv")

df_sample = df.sample(n=100, random_state=42).reset_index(drop=True)

print(df_sample.head())

                                         id  \
0  f00ae3c3929d829cd469ba4f229cc613b0766203   
1  9e451f79499e5c784222b3f237c6ae4829849d79   
2  dae58055bd50598b93a230aa3a58e0d2f519b536   
3  c05bda9b387ec8ae43803170b6f59b4b82505db9   
4  5c7493c6f28cfd58aa7b5f0e486e611307b4126d   

                                             article  \
0  Comedian Jenny Eclair travelled with her other...   
1  A woman of Arab and Jewish descent who was str...   
2  World No 1 Novak Djokovic has apologised to th...   
3  (CNN)ISIS on Wednesday released more than 200 ...   
4  Hillary Clinton’s security detail arrived at a...   

                                          highlights  
0  The comedian stayed with Flavours who offer a ...  
1  The federal government will give Shoshana Hebs...  
2  Novak Djokovic beat Andy Murray 7-6 4-6 6-0 in...  
3  Most of those released were women and children...  
4  Second modified, armored van spotted near Des ...  


Preprocessing

In [23]:
results = []

In [5]:
import re

def preprocess_article(text, max_chars=3500):
    text = re.sub(r"\s+", " ", text).strip()
    
    if len(text) <= max_chars:
        return text
    
    cut = text[:max_chars]
    
    last_period = cut.rfind(".")
    if last_period != -1:
        return cut[:last_period+1]
    
    return cut

In [ ]:
for i in range(len(df_sample)):
    article = df_sample.loc[i, "article"]
    reference = df_sample.loc[i, "highlights"]

    processed_article = preprocess_article(article)

    df_sample.loc[i, "processed_article"] = processed_article

print("Preprocessing complete.")

Preprocessing complete.


ROUGE Evaluation Function

In [15]:
from rouge_score import rouge_scorer

In [16]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(system_summary, reference_summary):
    scores = scorer.score(reference_summary, system_summary)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

Extractive Summarization (TextRank)

In [27]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

In [ ]:
# TextRank

for i in range(len(df_sample)):
    processed_article = df_sample.loc[i, "processed_article"]
    reference = df_sample.loc[i, "highlights"]

    parser = PlaintextParser.from_string(processed_article, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    summary_sentences = summarizer(parser.document, 3)

    textrank_summary = " ".join(str(sentence) for sentence in summary_sentences)

    scores = compute_rouge(textrank_summary, reference)

    results.append({
        "model": "textrank",
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

print("TextRank done!")


TextRank done!


Abstractive Summarization (BART and Pegasus)

In [9]:
from huggingface_hub import InferenceClient

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# BART

client = InferenceClient(
    "facebook/bart-large-cnn",
    token = os.environ.get("HF_TOKEN")
)

def bart_summarize(text):
    output = client.summarization(text)
    return output

In [ ]:

for i in range(len(df_sample)):
    processed_article = df_sample.loc[i, "processed_article"]
    reference = df_sample.loc[i, "highlights"]

    try:
        bart_output = bart_summarize(processed_article)
        bart_summary = bart_output["summary_text"]
    except Exception as e:
        print(f"BART error on sample {i}: {e}")
        bart_summary = ""  

    scores = compute_rouge(bart_summary, reference)

    results.append({
        "model": "bart",
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

print("BART done!")

BART done!


In [ ]:
# PEGASUS

import requests
import json
import time

API_URL = "https://router.huggingface.co/hf-inference/models/google/pegasus-cnn_dailymail"
headers = {
    "Authorization": f"Bearer {os.environ.get('HF_TOKEN')}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

def safe_trim(text, max_len):
    if len(text) <= max_len:
        return text
    
    cut = text[:max_len]
    last_period = cut.rfind(".")
    if last_period != -1:
        return cut[:last_period+1]
    return cut


def pegasus_summarizer(text, max_retries=5):
    current_text = text

    for attempt in range(1, max_retries + 1):
        print(f"\n🟦 PEGASUS ATTEMPT {attempt}/{max_retries}")

        timeout = 60 + attempt * 40
        print(f"⏳ Timeout set to: {timeout} seconds")

        payload = {"inputs": current_text}

        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.exceptions.Timeout:
            print("⚠️ Timeout reached — retrying...")
            continue

        print("🔵 STATUS CODE:", resp.status_code)

        if resp.status_code == 429:
            print("⚠️ Rate limited — waiting 10 seconds...")
            time.sleep(10)
            continue

        if resp.status_code >= 500:
            print("⚠️ Server error — retrying...")
            time.sleep(3)
            continue

        try:
            out = resp.json()
            return out
        except:
            print("⚠️ Non-JSON output — retrying with trimmed text...")
            current_text = safe_trim(current_text, int(len(current_text) * 0.8))
            continue

    return {"error": "failed_all_retries"}

In [ ]:
for i in range(len(df_sample)):
    processed_article = df_sample.loc[i, "processed_article"]
    reference = df_sample.loc[i, "highlights"]

    pegasus_output = pegasus_summarizer(processed_article)

    if isinstance(pegasus_output, dict) and "error" in pegasus_output:
        print(f"Pegasus error on sample {i}: {pegasus_output}")
        pegasus_summary = ""
    else:
        try:
            pegasus_summary = pegasus_output[0]["summary_text"]
        except:
            print(f"Pegasus unexpected format on sample {i}: {pegasus_output}")
            pegasus_summary = ""

    scores = compute_rouge(pegasus_summary, reference)

    results.append({
        "model": "pegasus",
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

print("Pegasus done!")



🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
🔵 STATUS CODE: 200

🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
⚠️ Timeout reached — retrying...

🟦 PEGASUS ATTEMPT 2/5
⏳ Timeout set to: 140 seconds
🔵 STATUS CODE: 504
⚠️ Server error — retrying...

🟦 PEGASUS ATTEMPT 3/5
⏳ Timeout set to: 180 seconds
🔵 STATUS CODE: 504
⚠️ Server error — retrying...

🟦 PEGASUS ATTEMPT 4/5
⏳ Timeout set to: 220 seconds
🔵 STATUS CODE: 200

🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
🔵 STATUS CODE: 200

🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
🔵 STATUS CODE: 200

🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
⚠️ Timeout reached — retrying...

🟦 PEGASUS ATTEMPT 2/5
⏳ Timeout set to: 140 seconds
🔵 STATUS CODE: 504
⚠️ Server error — retrying...

🟦 PEGASUS ATTEMPT 3/5
⏳ Timeout set to: 180 seconds
🔵 STATUS CODE: 504
⚠️ Server error — retrying...

🟦 PEGASUS ATTEMPT 4/5
⏳ Timeout set to: 220 seconds
🔵 STATUS CODE: 200

🟦 PEGASUS ATTEMPT 1/5
⏳ Timeout set to: 100 seconds
🔵 STATU

ROGUE Evaluation

In [ ]:
results_df = pd.DataFrame(results)

results_df.head()

,model,rouge1,rouge2,rougeL
0,textrank,0.222222,0.075000,0.160494
1,textrank,0.296296,0.149733,0.211640
2,textrank,0.300000,0.101010,0.190000
3,textrank,0.250000,0.163636,0.214286
4,textrank,0.346939,0.082474,0.163265


In [37]:
average_scores = results_df.groupby("model").mean()
average_scores

,rouge1,rouge2,rougeL
model,,,
bart,0.437131,0.195768,0.292264
pegasus,0.431852,0.205922,0.303456
textrank,0.308208,0.107340,0.189286
